# TP2 — Named Entity Recognition with a Transformer (BERT)

Adapted from `transformers_classification.py`, which fine-tunes BERT for **sentence**
classification. NER is **token** classification.

| | `transformers_classification.py` | this notebook |
|---|---|---|
| Head | `AutoModelForSequenceClassification` | **`AutoModelForTokenClassification`** |
| Data | CSV, one `review` + one `label` per row | CoNLL, one token + one tag per line |
| Tokenizer call | `tokenizer(texts, padding, truncation)` | `tokenizer(words, is_split_into_words=True)` + label alignment |
| Labels | one float per sentence | one tag id per token, `-100` where ignored |
| Loss | sigmoid + BCE (multi-label) | cross-entropy over 21 tags (the head does it) |
| Metric | `sklearn` f1 / roc_auc / accuracy | **`seqeval`** entity-level P / R / F1 |

What is kept from the script: the `AutoTokenizer` / `TrainingArguments` / `Trainer` structure,
the hyper-parameter names, and the train-then-evaluate-on-valid-and-test flow.

What is dropped: the `datasets`-based CSV preprocessing (`string_list_2_list`,
`preprocess_labels`, `encode`, `padding`) and the multi-label metric helpers. A CoNLL file
is already one token per line, so a small `torch.utils.data.Dataset` replaces all of it.

> Two API names in the script are out of date for `transformers` 4.57:
> `TrainingArguments(evaluation_strategy=...)` is now `eval_strategy`, and
> `Trainer(tokenizer=...)` is now `processing_class`.


## 0. Requirements

Runs unchanged on Google Colab or locally.

**On Colab:** set the runtime to a GPU first (*Runtime → Change runtime type → T4 GPU*),
then run the two cells below. The first installs the libraries; the second asks you to
upload `quaero_conll.zip` (the CoNLL files, 578 KB) and unpacks it.

**Locally:** both cells detect they are not on Colab and do nothing; the data is found at
`../TP_ISD2020/`.

In [ ]:
IN_COLAB = 'google.colab' in str(get_ipython())

if IN_COLAB:
    !pip install -q transformers accelerate seqeval
    print('installed')
else:
    print('not on Colab -- using the local environment')

In [ ]:
from pathlib import Path

# on Colab, upload quaero_conll.zip when prompted
if IN_COLAB and not Path('TP_ISD2020').exists():
    from google.colab import files
    print('Select quaero_conll.zip ...')
    files.upload()
    !unzip -q -o quaero_conll.zip

# the same notebook works from Workshop3/scripts/ locally and from /content on Colab
DATA = next(
    p for p in (Path('TP_ISD2020/QUAERO_FrenchMed'),
                Path('../TP_ISD2020/QUAERO_FrenchMed'))
    if p.exists()
)
print('data:', DATA.resolve())

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
from transformers import (
    AutoModelForTokenClassification,   
    AutoTokenizer,
    DataCollatorForTokenClassification,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
    set_seed,
)
from seqeval.metrics import classification_report, precision_score, recall_score, f1_score

set_seed(42)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device, '| torch', torch.__version__)
if device == 'cuda':
    print('gpu   :', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU -- this takes hours on CPU. On Colab: Runtime > Change runtime type > T4 GPU')


## 1. Configuration

`camembert-base` is the standard French BERT. A biomedical French model such as
`Dr-BERT/DrBERT-7GB` can be dropped in by changing `MODEL_NAME` alone.


In [ ]:
MODEL_NAME = 'camembert-base'
DATASETS = ['EMEA', 'MEDLINE']      # the two annotated sub-corpora

# the hyper-parameters of the original script
MAX_LEN = 128
TRAIN_BATCH_SIZE = 8
EPOCHS = 20            # 5 (the script's default) stops while dev F1 is still climbing
LEARNING_RATE = 3e-5
EARLY_STOPPING = 3     # stop after 3 epochs without a dev-F1 improvement

OUT_DIR = Path('transformer_runs')  # Trainer checkpoints
LABEL_PAD = -100                   # ignored by the loss and by the metrics


## 2. Loading the CoNLL data

Same reader as the CNN and LSTM notebooks: column 2 is the token, column 5 is the NER tag,
an empty line separates two sentences.


In [ ]:
def read_conll(path):
    """Read a CoNLL file as (sentences, tags), both lists of lists of strings."""
    sentences, tags = [], []
    tokens, labels = [], []

    for line in path.read_text(encoding='utf-8').splitlines():
        columns = line.split()
        if not columns:                  # empty line -> end of a sentence
            if tokens:
                sentences.append(tokens)
                tags.append(labels)
                tokens, labels = [], []
            continue
        tokens.append(columns[1])        # column 2 = the token
        labels.append(columns[4])        # column 5 = the NER tag

    if tokens:
        sentences.append(tokens)
        tags.append(labels)

    return sentences, tags


def load_split(dataset, split):
    return read_conll(DATA / dataset / f'{dataset}{split}_layer1_ID.conll')


corpora = {
    name: {split: load_split(name, split) for split in ('train', 'dev', 'test')}
    for name in DATASETS
}

display(pd.DataFrame({
    name: {f'{split} sentences': len(splits[split][0]) for split in splits}
    for name, splits in corpora.items()
}))


### The tag set

Built from the training sets of both datasets so that a single label mapping is shared,
which keeps the two runs comparable.


In [ ]:
TAGS = sorted({
    tag
    for splits in corpora.values()
    for tags in splits['train'][1]
    for tag in tags
})
label2id = {t: i for i, t in enumerate(TAGS)}
id2label = {i: t for t, i in label2id.items()}

print(f'{len(TAGS)} tags:', ' '.join(TAGS))


## 3. Tokenization and label alignment

This is the real work of adapting the script. BERT uses sub-word units, so one word can
become several tokens and the labels no longer line up one to one:

```
word     Prialt            est   une   solution
subword  ▁Pri  al  t       ▁est  ▁une  ▁solution
```

The convention used here is the standard one: a word's tag goes on its **first** sub-word,
and every continuation sub-word plus the special tokens (`<s>`, `</s>`, padding) get
`-100`, which `AutoModelForTokenClassification` excludes from the loss. `word_ids()` from
the fast tokenizer gives the sub-word → word mapping.

At evaluation time the `-100` positions are dropped again, which recovers exactly one
prediction per original word — what `seqeval` needs.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def encode(words, tags):
    """Tokenize one pre-split sentence and align its tags onto the sub-words."""
    encoding = tokenizer(
        words,
        is_split_into_words=True,
        truncation=True,
        max_length=MAX_LEN,
    )

    labels = []
    previous_word = None
    for word_index in encoding.word_ids():
        if word_index is None:              # <s>, </s>
            labels.append(LABEL_PAD)
        elif word_index != previous_word:   # first sub-word of a word
            labels.append(label2id[tags[word_index]])
        else:                               # continuation sub-word
            labels.append(LABEL_PAD)
        previous_word = word_index

    encoding['labels'] = labels
    return encoding


A worked example, to check the alignment is right:


In [ ]:
words, tags = corpora['EMEA']['train'][0][2], corpora['EMEA']['train'][1][2]
example = encode(words, tags)

display(pd.DataFrame({
    'sub-word': tokenizer.convert_ids_to_tokens(example['input_ids']),
    'word_id':  [('-' if w is None else w) for w in example.word_ids()],
    'label':    [('ignored' if l == LABEL_PAD else id2label[l]) for l in example['labels']],
}))


### Dataset and collator

`DataCollatorForTokenClassification` pads each batch to its longest sentence and pads the
labels with `-100`, so no manual `padding()` function is needed.


In [ ]:
class ConllDataset(Dataset):
    """A CoNLL split, tokenized and label-aligned."""

    def __init__(self, sentences, tags):
        self.examples = [encode(w, t) for w, t in zip(sentences, tags)]

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, i):
        return dict(self.examples[i])


collator = DataCollatorForTokenClassification(tokenizer, label_pad_token_id=LABEL_PAD)

datasets = {
    name: {split: ConllDataset(*splits[split]) for split in splits}
    for name, splits in corpora.items()
}

for name, splits in datasets.items():
    print(name, {split: len(d) for split, d in splits.items()})


## 4. Metrics

`compute_metrics` replaces the script's `multi_label_metrics`. The `-100` positions are
removed, the ids are mapped back to tag strings, and `seqeval` scores whole entities.


In [ ]:
def decode(predictions, labels):
    """Drop the ignored positions and map ids back to tag strings."""
    y_true, y_pred = [], []
    for pred_row, label_row in zip(predictions, labels):
        keep = label_row != LABEL_PAD
        y_true.append([id2label[i] for i in label_row[keep]])
        y_pred.append([id2label[i] for i in pred_row[keep]])
    return y_true, y_pred


def compute_metrics(eval_prediction):
    logits, labels = eval_prediction
    predictions = logits.argmax(axis=-1)
    y_true, y_pred = decode(predictions, labels)

    return {
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall':    recall_score(y_true, y_pred, zero_division=0),
        'f1':        f1_score(y_true, y_pred, zero_division=0),
    }


## 5. Fine-tuning

`AutoModelForTokenClassification` is the one line the lab explicitly asks to change. It
puts a linear layer over every token's hidden state and applies cross-entropy with
`ignore_index=-100` internally, so no objective function has to be written by hand.

The best epoch is selected on the **dev** set by entity F1, not by loss.

> **On the number of epochs.** The script defaults to `EPOCHS = 5`, which is fine for a
> large dataset but far too few here: with 706 training sentences and batch 8 that is only
> 445 optimiser steps. A first run at 5 epochs scored 0 dev F1 for its first *two* epochs
> and was still gaining ~2.5 F1 points on the last one, so it measured an unconverged
> model. We therefore train up to 20 epochs and let early stopping on dev F1 decide.

In [ ]:
def run_experiment(dataset, epochs=EPOCHS, lr=LEARNING_RATE, verbose=True):
    """Fine-tune the model on one dataset and evaluate it on dev and test."""
    splits = datasets[dataset]

    model = AutoModelForTokenClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(TAGS),
        id2label=id2label,
        label2id=label2id,
    )

    args = TrainingArguments(
        output_dir=str(OUT_DIR / f"{MODEL_NAME.replace('/', '_')}_{dataset}"),
        eval_strategy='epoch',          # 'evaluation_strategy' in older versions
        save_strategy='epoch',
        learning_rate=lr,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=TRAIN_BATCH_SIZE,
        num_train_epochs=epochs,
        load_best_model_at_end=True,
        metric_for_best_model='f1',    # pick the best epoch on entity F1
        greater_is_better=True,
        save_total_limit=1,
        logging_strategy='epoch',
        fp16=torch.cuda.is_available(),   # mixed precision when a GPU is present
        report_to='none',
        seed=42,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=splits['train'],
        eval_dataset=splits['dev'],
        processing_class=tokenizer,     # 'tokenizer' in older versions
        data_collator=collator,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING)],
    )

    trainer.train()

    dev = trainer.evaluate(splits['dev'])
    test = trainer.evaluate(splits['test'])

    # the per-entity breakdown on test
    output = trainer.predict(splits['test'])
    y_true, y_pred = decode(output.predictions.argmax(axis=-1), output.label_ids)
    report = classification_report(y_true, y_pred, digits=4, zero_division=0)

    if verbose:
        print(f'\n===== {dataset} / {MODEL_NAME} =====')
        print(report)

    return {'dev': dev, 'test': test, 'report': report}


### Run both datasets


In [ ]:
runs = {name: run_experiment(name) for name in DATASETS}


## 6. Results


In [ ]:
transformer_scores = pd.DataFrame({
    name: {
        'precision': r['test']['eval_precision'],
        'recall':    r['test']['eval_recall'],
        'f1':        r['test']['eval_f1'],
        'dev f1':    r['dev']['eval_f1'],
    }
    for name, r in runs.items()
}).T.round(4)

display(transformer_scores)

OUT_DIR.mkdir(exist_ok=True)
transformer_scores.to_csv(OUT_DIR / 'results_transformer.csv')
print(f"saved {OUT_DIR / 'results_transformer.csv'}")


### Per-entity detail


In [ ]:
for name, r in runs.items():
    print(f'===== {name} =====')
    print(r['report'])


## 7. Comparison with the CNN and the LSTM

The CNN and LSTM figures below are the best configuration of each, on EMEA, taken from
`cnn.ipynb` and `lstm.ipynb` (same splits, same `seqeval` entity-level metric).


In [ ]:
baselines = pd.DataFrame([
    {'model': 'CNN  (random embeddings)',        'precision': 0.4106, 'recall': 0.1954, 'f1': 0.2648},
    {'model': 'LSTM (random embeddings)',        'precision': 0.3770, 'recall': 0.1581, 'f1': 0.2227},
    {'model': 'CNN  (press skipgram, tuned)',    'precision': 0.4623, 'recall': 0.4072, 'f1': 0.4330},
    {'model': 'LSTM (press fastText, frozen)',   'precision': 0.4884, 'recall': 0.3798, 'f1': 0.4273},
]).set_index('model')

transformer_emea = pd.DataFrame([{
    'model':     f'BERT ({MODEL_NAME}, fine-tuned)',
    'precision': runs['EMEA']['test']['eval_precision'],
    'recall':    runs['EMEA']['test']['eval_recall'],
    'f1':        runs['EMEA']['test']['eval_f1'],
}]).set_index('model')

comparison = pd.concat([baselines, transformer_emea]).round(4)
display(comparison.sort_values('f1', ascending=False))


## 8. Answers to the lab questions

*(fill in from the tables above once the notebook has been run)*

**Which model provides the best performance on each dataset?**
See section 6 for the Transformer on EMEA and MEDLINE, and section 7 for the comparison
against the CNN and the LSTM on EMEA.

**Do you see any differences between the embeddings learned on the two TP1 corpora?**
Yes, and in both directions. On the intrinsic task (`main.ipynb`) the two corpora learn
different *senses* of the same word: *jaune* is a cycling jersey in the press corpus and a
fever in the medical one. On NER, the press embeddings won despite being out of domain
(CNN F1 0.433 vs 0.367 for the medical ones), because the medical corpus has only 52K
tokens and its vectors are undertrained — corpus size mattered more than domain match.

**Compare these results with a Transformer model.**
See section 7. The Transformer needs no pretrained word vectors at all: its sub-word
tokenizer has no out-of-vocabulary case, and it arrives already pretrained on a large
French corpus, which is exactly what the word-level models lacked.
